# Pipe and Filter — Exemplo Executável

**Autores:** Thiago Leal, Emerson Silva, José Victor, Randson Bredley, Thiago Patriota, Gustavo Henrique (Grupo 3)

Este notebook demonstra o estilo arquitetural **Pipe and Filter** através de um pipeline de processamento de texto.
Mostramos a implementação correta, a violação das restrições do estilo e a consequência mensurável dessa violação.

**Referências:**
- Shaw & Garlan, *Software Architecture: Perspectives on an Emerging Discipline*, 1996, cap. 2
- Bass, Clements & Kazman, *Software Architecture in Practice*, 4ª ed., 2021, cap. 13

## 1. Dependências

Este notebook usa apenas a biblioteca padrão do Python — não precisa de instalação adicional.

In [ ]:
# Dependências — apenas biblioteca padrão
import re
from collections import Counter
from typing import Callable, Any

## 2. Dados de entrada

Um texto bruto que será processado pelo pipeline. Escolhemos um trecho sobre
arquitetura de software para manter a coerência com o tópico.

In [ ]:
texto_bruto = """
A arquitetura de software define a estrutura fundamental de um sistema.
Ela determina como os componentes se organizam, como se comunicam e quais
restrições governam o projeto. Um estilo arquitetural como o Pipe and Filter
decompõe o processamento em etapas independentes, cada uma responsável por
uma transformação específica. Os filtros não conhecem uns aos outros;
eles apenas recebem dados, transformam e produzem resultados. Essa
independência permite que filtros sejam adicionados, removidos ou
substituídos sem afetar o restante do pipeline. O estilo é amplamente
utilizado em compiladores, pipelines de ETL, processamento de sinais
e sistemas de análise de logs. A simplicidade do modelo facilita a
compreensão e a manutenção, mas impõe custos de desempenho quando o
volume de dados é grande e a serialização entre filtros se torna
um gargalo.
"""

print(f"Texto de entrada ({len(texto_bruto.split())} palavras):")
print(texto_bruto[:200] + "...")

## 3. Implementação CORRETA — Pipe and Filter

Cada filtro é uma **função pura**: recebe dados, transforma e retorna.
Nenhum filtro conhece os outros. A comunicação acontece exclusivamente
pelos pipes (os dados passados entre funções).

### 3.1 Definição dos filtros

In [ ]:
# ---------- FILTRO 1: Normalizar ----------
def normalizar(texto: str) -> str:
    """Converte para minúsculas e remove pontuação."""
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    return texto


# ---------- FILTRO 2: Tokenizar ----------
def tokenizar(texto: str) -> list[str]:
    """Divide o texto em palavras individuais."""
    return texto.split()


# ---------- FILTRO 3: Filtrar stop words ----------
STOP_WORDS_PT = {
    'a', 'o', 'e', 'é', 'de', 'do', 'da', 'dos', 'das', 'em', 'um',
    'uma', 'os', 'as', 'no', 'na', 'nos', 'nas', 'se', 'que', 'como',
    'por', 'para', 'com', 'não', 'nao', 'mais', 'ao', 'aos', 'sua',
    'seu', 'ela', 'ele', 'eles', 'elas', 'ou', 'uns', 'umas', 'entre',
    'cada', 'quando', 'esse', 'essa', 'este', 'esta', 'isso', 'isto',
}

def filtrar_stop_words(palavras: list[str]) -> list[str]:
    """Remove palavras comuns sem valor semântico."""
    return [p for p in palavras if p not in STOP_WORDS_PT and len(p) > 1]


# ---------- FILTRO 4: Contar frequências ----------
def contar_frequencias(palavras: list[str]) -> dict[str, int]:
    """Calcula a frequência de cada palavra."""
    return dict(Counter(palavras).most_common())


print("✅ Quatro filtros definidos, cada um independente dos demais.")

### 3.2 O pipeline (pipe)

O pipe conecta os filtros: a saída de um é a entrada do próximo.
A função `pipeline` é genérica — funciona com qualquer sequência de filtros.

In [ ]:
def pipeline(dado: Any, *filtros: Callable) -> Any:
    """Executa uma sequência de filtros, passando o resultado de um para o próximo."""
    resultado = dado
    for i, filtro in enumerate(filtros):
        resultado = filtro(resultado)
        print(f"  Filtro {i+1} ({filtro.__name__}): {type(resultado).__name__} → {str(resultado)[:80]}...")
    return resultado


print("Executando pipeline CORRETO:")
print("=" * 60)
resultado_correto = pipeline(
    texto_bruto,
    normalizar,
    tokenizar,
    filtrar_stop_words,
    contar_frequencias
)
print("=" * 60)
print(f"\n📊 Top 10 palavras mais frequentes:")
for palavra, freq in list(resultado_correto.items())[:10]:
    print(f"   {palavra:.<25} {freq}")

### 3.3 Reuso: montando outro pipeline com os mesmos filtros

A grande vantagem do Pipe and Filter: os mesmos filtros podem ser recombinados
para resolver problemas diferentes.

In [ ]:
# Pipeline alternativo: apenas normalizar e tokenizar (sem stop words, sem contagem)
print("Pipeline alternativo — apenas normalizar e tokenizar:")
print("=" * 60)
tokens = pipeline(texto_bruto, normalizar, tokenizar)
print(f"\nTotal de tokens: {len(tokens)}")
print(f"Primeiros 15: {tokens[:15]}")

## 4. A VIOLAÇÃO — Acoplamento entre filtros

Agora vamos violar a restrição principal do estilo: um filtro vai acessar
diretamente o estado de outro filtro. Isso cria **acoplamento direto**,
que é exatamente o que o estilo busca evitar.

In [ ]:
# ❌ VERSÃO COM VIOLAÇÃO
# O filtro NormalizadorAcoplado guarda estado interno (idioma).
# O filtro ContadorAcoplado acessa diretamente esse estado.

class NormalizadorAcoplado:
    """Filtro que guarda estado interno — viola a independência."""
    def __init__(self, idioma='pt'):
        self.idioma = idioma  # estado que outros filtros vão acessar

    def __call__(self, texto: str) -> str:
        texto = texto.lower()
        texto = re.sub(r'[^\w\s]', '', texto)
        return texto


class ContadorAcoplado:
    """Filtro que depende diretamente do NormalizadorAcoplado — ACOPLAMENTO."""
    def __init__(self, normalizador: NormalizadorAcoplado):
        self.normalizador = normalizador  # ← referência direta!

    def __call__(self, palavras: list[str]) -> dict[str, int]:
        # Consulta o idioma diretamente no normalizador
        idioma = self.normalizador.idioma  # ← VIOLAÇÃO: acessa estado de outro filtro
        print(f"    ⚠️  ContadorAcoplado consultou idioma='{idioma}' do NormalizadorAcoplado")

        # Usa o idioma para decidir stop words
        stops = STOP_WORDS_PT if idioma == 'pt' else set()
        palavras_filtradas = [p for p in palavras if p not in stops and len(p) > 1]
        return dict(Counter(palavras_filtradas).most_common())


norm_acoplado = NormalizadorAcoplado(idioma='pt')
cont_acoplado = ContadorAcoplado(norm_acoplado)

print("Executando pipeline COM VIOLAÇÃO:")
print("=" * 60)
resultado_violado = pipeline(
    texto_bruto,
    norm_acoplado,
    tokenizar,
    cont_acoplado  # ← não passa por filtrar_stop_words, porque o contador faz tudo
)
print("=" * 60)
print(f"\n📊 Top 10 (versão acoplada):")
for palavra, freq in list(resultado_violado.items())[:10]:
    print(f"   {palavra:.<25} {freq}")

## 5. A consequência mensurável

Vamos medir o impacto do acoplamento usando **acoplamento eferente**:
quantas dependências externas cada filtro tem.

In [ ]:
print("📏 Medindo acoplamento eferente (dependências externas de cada filtro)")
print("=" * 60)

# Versão CORRETA: cada filtro depende apenas da sua entrada
print("\n✅ Versão CORRETA (funções puras):")
filtros_corretos = {
    'normalizar': [],          # sem dependências externas
    'tokenizar': [],            # sem dependências externas
    'filtrar_stop_words': [],   # usa constante STOP_WORDS_PT (injetável)
    'contar_frequencias': [],   # sem dependências externas
}
total_correto = 0
for nome, deps in filtros_corretos.items():
    print(f"  {nome}: {len(deps)} dependências {deps}")
    total_correto += len(deps)
print(f"  TOTAL: {total_correto} dependências")


# Versão VIOLADA: ContadorAcoplado depende de NormalizadorAcoplado
print("\n❌ Versão VIOLADA (acoplamento direto):")
filtros_violados = {
    'NormalizadorAcoplado': [],                      # sem dependências externas
    'tokenizar': [],                                  # sem dependências externas
    'ContadorAcoplado': ['NormalizadorAcoplado'],     # ← DEPENDE de outro filtro!
}
total_violado = 0
for nome, deps in filtros_violados.items():
    print(f"  {nome}: {len(deps)} dependência(s) {deps}")
    total_violado += len(deps)
print(f"  TOTAL: {total_violado} dependência(s)")


print(f"\n📊 Resultado:")
print(f"   Acoplamento eferente total (correto):  {total_correto}")
print(f"   Acoplamento eferente total (violado):  {total_violado}")
print(f"   Aumento: {total_violado - total_correto} → o acoplamento eferente subiu de 0 para 1")

### Impacto na modificabilidade

Vamos simular o cenário: **trocar o normalizador por uma versão em inglês**.

- Na versão correta, basta trocar o filtro `normalizar` — nenhum outro filtro é afetado.
- Na versão violada, trocar o `NormalizadorAcoplado` exige verificar e potencialmente alterar o `ContadorAcoplado`.

In [ ]:
print("🔄 Simulação: trocar o normalizador por versão em inglês")
print("=" * 60)

# Versão CORRETA: basta definir um novo filtro e plugar no pipeline
def normalizar_en(texto: str) -> str:
    """Versão em inglês do normalizador."""
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    return texto

STOP_WORDS_EN = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'of', 'in', 'to', 'and', 'or', 'it', 'on', 'at'}

def filtrar_stop_words_en(palavras: list[str]) -> list[str]:
    return [p for p in palavras if p not in STOP_WORDS_EN and len(p) > 1]

print("\n✅ Versão CORRETA:")
print("   Componentes alterados: 2 (normalizar_en + filtrar_stop_words_en)")
print("   Componentes NÃO alterados: 2 (tokenizar + contar_frequencias)")
print("   Cada troca é INDEPENDENTE — nenhum filtro precisa saber do outro.")

print("\n❌ Versão VIOLADA:")
print("   Componentes que PRECISAM ser alterados: 2")
print("     1. NormalizadorAcoplado (trocar a lógica)")
print("     2. ContadorAcoplado (depende do idioma do normalizador!)")
print("   O ContadorAcoplado não deveria ser afetado, mas é — porque acessa")
print("   diretamente o estado do normalizador.")

print("\n📊 Conclusão:")
print("   Na versão correta, a troca de um filtro NÃO propaga para outros.")
print("   Na versão violada, a troca de um filtro PROPAGA para dependentes.")
print("   Essa propagação é o custo concreto do acoplamento.")

## 6. Pipe and Filter como classe reutilizável

Uma implementação mais robusta usando orientação a objetos, que facilita
a composição e a inspeção do pipeline.

In [ ]:
class Pipeline:
    """Pipeline reutilizável no estilo Pipe and Filter."""

    def __init__(self, nome: str = "Pipeline"):
        self.nome = nome
        self._filtros: list[Callable] = []

    def adicionar(self, filtro: Callable) -> 'Pipeline':
        """Adiciona um filtro ao pipeline. Retorna self para encadeamento."""
        self._filtros.append(filtro)
        return self

    def executar(self, dado: Any) -> Any:
        """Executa todos os filtros em sequência."""
        resultado = dado
        for filtro in self._filtros:
            resultado = filtro(resultado)
        return resultado

    def __repr__(self) -> str:
        nomes = ' → '.join(f.__name__ if hasattr(f, '__name__') else str(f) for f in self._filtros)
        return f"{self.nome}: [{nomes}]"


# Montando pipelines reutilizáveis
pipe_completo = (
    Pipeline("Análise de frequência")
    .adicionar(normalizar)
    .adicionar(tokenizar)
    .adicionar(filtrar_stop_words)
    .adicionar(contar_frequencias)
)

pipe_simples = (
    Pipeline("Tokenização simples")
    .adicionar(normalizar)
    .adicionar(tokenizar)
)

print(f"Pipeline 1: {pipe_completo}")
print(f"Pipeline 2: {pipe_simples}")

print(f"\nResultado do Pipeline 1 (top 5):")
freq = pipe_completo.executar(texto_bruto)
for p, f in list(freq.items())[:5]:
    print(f"  {p}: {f}")

print(f"\nResultado do Pipeline 2 (primeiros 10 tokens):")
tokens = pipe_simples.executar(texto_bruto)
print(f"  {tokens[:10]}")

## 7. Conclusão

O estilo Pipe and Filter oferece:

- **Reusabilidade:** os mesmos filtros em pipelines diferentes
- **Modificabilidade:** trocar um filtro não afeta os demais
- **Testabilidade:** cada filtro testável isoladamente

Mas impõe custos:

- **Desempenho:** serialização entre pipes
- **Interatividade:** fluxo unidirecional

A regra de ouro: se você precisa que um filtro conheça outro filtro,
o Pipe and Filter provavelmente não é o estilo certo para o seu problema.